# Stage 02b — Postprocessing (wizard-driven, 518×518 for DINOv2)

Reads the **02a_preprocessing output** (newest
`preprocessed/preprocessed_patents_<batch>_<timestamp>.xlsx` under
`paths.html_review_exports` — flat "Review" sheet, IMGPATH-resolved,
validation Rules A–D applied; falls back to the legacy
`reviewed_patents_<batch>.xlsx` in `paths.wizard_resolved_dir` if 02a hasn't
run), takes every **approved** figure, applies the reviewer's
`rotation_deg`, resizes with aspect ratio preserved and pads to a
518×518 square with a **background-aware fill** (`src/processor.py:adaptive_pad_resize`):

- the reviewer's `bgSty` label decides solid-vs-reflect when present
  (*Solid Fill* → flat median-colour fill; *Shaded/Gradient* / *Grid/Pattern* → mirror-extend);
- unlabelled images fall back to the border-ring dominant-colour rule.

Outputs (mirror the `matched/` ↔ `data/matched/` convention):

```
processed/<batch>/all/<patent_dir>/<orig_name>.png     every approved figure
processed/<batch>/main/<patent>_arch<N>_fig<K>.png     flat canonical set (isMain)
data/processed/<batch>/processing_manifest_<batch>.csv the contract with DINOv2_eVTOL_frozen_Analysis
```


In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from src.config_loader import load_config
import src.processor as proc

cfg = load_config()
sheet_name  = "Batch_01"          # <- the batch to process
matched_dir = Path(cfg["paths"]["matched"]) / sheet_name

# ── Input: prefer 02a_preprocessing's output ────────────────────────────────
# 02a writes preprocessed_patents_<batch>_<timestamp>.xlsx (flat "Review"
# sheet, IMGPATH-resolved, validation rules applied) under
# html_review_exports/preprocessed/. Pick the newest for this batch; fall
# back to the legacy wizard_resolved_dir convention if 02a hasn't run.
preprocessed_dir = Path(cfg["paths"]["html_review_exports"]) / "preprocessed"
candidates = sorted(preprocessed_dir.glob(f"preprocessed_patents_{sheet_name}_*.xlsx"))
if candidates:
    REVIEWED_XLSX = candidates[-1]   # timestamped names sort chronologically
    print(f"using 02a output ({len(candidates)} run(s) found, newest picked)")
else:
    REVIEWED_XLSX = Path(cfg["paths"]["wizard_resolved_dir"]) / f"reviewed_patents_{sheet_name}.xlsx"
    print("⚠ no 02a output found — falling back to legacy wizard_resolved_dir export")

print(f"batch         : {sheet_name}")
print(f"input export  : {REVIEWED_XLSX}")
print(f"processed out : {Path(cfg['paths']['processed']) / sheet_name}")
print(f"manifest out  : {Path(cfg['paths']['data_processed']) / sheet_name}")
print(f"target_size   : {cfg['processing']['target_size']}   pad mode: {cfg['processing']['pad_color_mode']}")


---
## Finalize this batch

Run after you've finished human review for this batch in
`UI_for_taxonomy_caracterization_10.0.html`: load `ml_predict_labels_<batch>.xlsx`
via the "Load Batch" xlsx input, review each patent, then click "Export batch"
(or let it auto-export on the last patent). That downloads
`reviewed_patents_<sheet_name>.xlsx` to your browser's download folder
(`cfg["paths"]["html_review_exports"]`, default `~/Downloads`) — the cell below
reads approval status (`T1 → isApproved`) directly from that file, no manual
`batches.xlsx` editing required.

In [ ]:
# ── Copy approved files → reviewed/{company}/{prototype}/{patent_id}/ ──────
# Approval status comes from the input export's T1/isApproved field per
# patent (REVIEWED_XLSX from the first cell — 02a's preprocessed output when
# available, so Rule D duplicate-inherited approvals are included).
# company_canonical/prototype_label still come from batches.xlsx, since the
# wizard export doesn't carry that grouping metadata.
import shutil
from pathlib import Path
import pandas as pd

if not REVIEWED_XLSX.exists():
    raise FileNotFoundError(
        f"{REVIEWED_XLSX} not found. Run 02a_preprocessing (or export from the "
        f"wizard) first."
    )

review_rows = pd.read_excel(REVIEWED_XLSX, sheet_name="Review")

def _is_true(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() == "true"

approval_rows = review_rows[(review_rows["Section"] == "T1") & (review_rows["Field"] == "isApproved")]
approved_ids = set(
    approval_rows.loc[approval_rows["Value"].apply(_is_true), "Patent_ID"].astype(str)
)
print(f"{len(approved_ids)} patent(s) marked approved in {REVIEWED_XLSX.name}.")

batches_path  = Path(cfg["paths"]["data"]) / "batches.xlsx"
batch_df_live = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
approved = batch_df_live[batch_df_live["patent_id"].astype(str).isin(approved_ids)]
print(f"{len(approved)}/{len(batch_df_live)} patents in {sheet_name} marked approved (via wizard export).")

reviewed_root = Path(cfg["paths"]["reviewed"])
copied, missing = 0, 0
for _, row in approved.iterrows():
    pid       = str(row["patent_id"]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()
    src_dir   = matched_dir / pid
    dst_dir   = reviewed_root / company / prototype / pid

    if not src_dir.exists():
        print(f"  ⚠ missing matched/ folder for {pid} — skipped")
        missing += 1
        continue
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    copied += 1

print(f"\nCopied: {copied}   Missing source: {missing}")


In [ ]:
# ── Collect batch outputs → <sheet_name_lower>/ ────────────────────────────
# Copies this batch's xlsx outputs (run_stage01's source_patents_<batch>.xlsx
# and ml_predict_labels_<batch>.xlsx, both written under data_matched/<sheet_name>/
# — there are no per-patent label JSONs in the current pipeline, see
# reviewer.py's module docstring) + any review HTML exports into a flat
# top-level folder for your own validation/inspection. Purely a convenience
# snapshot — skip once you trust the pipeline and are running all batches
# end-to-end without spot-checking each one individually.
import shutil
from pathlib import Path

BATCH_OUT = Path(cfg["paths"]["base"]) / sheet_name.lower()
BATCH_OUT.mkdir(parents=True, exist_ok=True)

# 1. per-batch xlsx outputs written by run_stage01()
batch_data_dir = Path(cfg["paths"]["data_matched"]) / sheet_name
n_xlsx = 0
for f in batch_data_dir.glob("*.xlsx"):
    shutil.copy2(f, BATCH_OUT / f.name)
    n_xlsx += 1

# 2. review HTML exports for this batch, if any were generated
html_src = Path(cfg["paths"]["data"]) / "review_html" / sheet_name
html_dst = BATCH_OUT / "review_html"
n_html = 0
if html_src.exists():
    shutil.copytree(html_src, html_dst, dirs_exist_ok=True)
    n_html = len(list(html_dst.glob("*")))

print(f"Collected {n_xlsx} xlsx file(s) and {n_html} HTML file(s) → {BATCH_OUT}")


---
## Run Stage 02

Pre-flight first (does the export exist, how many approved images, any missing
files), then `process_batch` writes `all/`, `main/` and the manifest.


In [ ]:
# ── Pre-flight: what will be processed? ─────────────────────────────────────
images = proc.load_review_images(REVIEWED_XLSX)
images["file_exists"] = images["Image_Path"].map(lambda p: Path(str(p)).exists())

n_missing = int((images["approved"] & ~images["file_exists"]).sum())
print(f"{len(images)} images in export | approved: {int(images['approved'].sum())} "
      f"(mains: {int((images['approved'] & images['is_main']).sum())}) | "
      f"rotated: {int((images['approved'] & images['rotation'].ne(0)).sum())}")
if n_missing:
    print(f"⚠ {n_missing} approved images MISSING on disk — did you run "
          f"scripts/resolve_image_paths.py on the export?")
    display(images.loc[images['approved'] & ~images['file_exists'], 'Image_Path'])
display(images.loc[images["approved"], ["bgSty", "bgCol", "acCol"]]
        .apply(lambda s: s.value_counts(dropna=False)).fillna(0).astype(int))


In [ ]:
# ── Process the batch ───────────────────────────────────────────────────────
# reviewed_xlsx=REVIEWED_XLSX so it consumes 02a's preprocessed output (or
# the legacy file, whichever the first cell resolved). force=True to reprocess.
manifest = proc.process_batch(sheet_name, cfg, reviewed_xlsx=REVIEWED_XLSX)


In [ ]:
# ── QC: fill decisions, seams, label agreement ──────────────────────────────
done = manifest[manifest["processed"]]

print("fill methods:", dict(done["fill_method"].value_counts()))
print("skip reasons:", dict(manifest.loc[~manifest["processed"], "skip_reason"].value_counts()))
print("\nseam std (0 = perfectly flat border after padding):")
display(done["seam_std"].describe().round(2).to_frame().T)

mismatch = done[done["fill_vs_label_ok"] == False]  # noqa: E712 — NaN-safe
if len(mismatch):
    print(f"⚠ {len(mismatch)} images where the painted fill disagrees with the bgCol label:")
    display(mismatch[["patent_id", "fig_key", "bg_col", "fill_color_hex",
                      "dominant_frac", "src_path"]])
else:
    print("✓ every painted fill agrees with its bgCol label")

print("\n10 worst seams (eyeball these):")
display(done.nlargest(10, "seam_std")[["patent_id", "fig_key", "fill_method",
                                       "fill_color_hex", "seam_std", "dst_all"]])


In [ ]:
# ── Visual check: original vs processed (mains first, then worst seams) ─────
import matplotlib.pyplot as plt
from PIL import Image

show = pd.concat([done[done["is_main"]].head(4),
                  done.nlargest(4, "seam_std")]).drop_duplicates("src_path")
fig, axes = plt.subplots(2, len(show), figsize=(3.2 * len(show), 7))
for j, (_, r) in enumerate(show.iterrows()):
    axes[0, j].imshow(Image.open(r["src_path"]))
    axes[0, j].set_title(f"{r['base_patent_id']}\nfig {r['fig_key']}", fontsize=8)
    axes[1, j].imshow(Image.open(r["dst_all"]))
    axes[1, j].set_title(f"{r['fill_method']} {r['fill_color_hex']}\nseam {r['seam_std']}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_ylabel("original"); axes[1, 0].set_ylabel("processed 518²")
plt.tight_layout(); plt.show()
